# 04 — PySpark Assembly Pipeline

Load raw NDVI CSVs, join daily PRISM weather on date, aggregate to weekly
tile means, compute the NDVI anomaly target, pivot to wide format, and join
static terrain + soil features. Saves a Parquet dataset ready for MLlib.

**Join order:**
1. Daily NDVI × daily weather → broadcast join on `date`
2. Weekly aggregation: NDVI mean/std, ppt sum, tmax/tmean/vpdmax means
3. Pivot to one row per (tile_id, year) with weekly feature columns
4. Broadcast join terrain (per tile) + soil (per tile)

**Prerequisites:** Java 17 (`sudo apt install -y openjdk-17-jdk`), python3.10 kernel.

In [1]:
import os, glob

os.environ.setdefault('JAVA_HOME', '/usr/lib/jvm/java-17-openjdk-amd64')
os.environ['PYSPARK_PYTHON']        = '/home/simonhans/anaconda3/envs/GrapeExpectationsML/bin/python'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/home/simonhans/anaconda3/envs/GrapeExpectationsML/bin/python'

NDVI_DIR  = '../data/ndvi/tiles_100m2'
TERRAIN   = '../data/terrain_100m2.csv'
SOIL      = '../data/soil_100m2.csv'
WEATHER   = '../data/weather_daily.csv'   # daily PRISM, single station, 2016-2025
OUT_DIR   = '../data/spark/assembled'

os.makedirs(OUT_DIR, exist_ok=True)

In [2]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName('GrapeExpectations-v2')
    .master('local[*]')
    .config('spark.driver.memory', '8g')
    .config('spark.sql.shuffle.partitions', '200')
    .getOrCreate())

spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/19 22:15:08 WARN Utils: Your hostname, office, resolves to a loopback address: 127.0.1.1; using 192.168.86.43 instead (on interface wlo1)
26/05/19 22:15:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/19 22:15:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1


In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType

raw_schema = StructType([
    StructField('tile_id', LongType(),   True),
    StructField('date',    StringType(), True),
    StructField('ndvi',    DoubleType(), True),
])

csv_files = glob.glob(os.path.join(NDVI_DIR, 'ndvi_100m2_*.csv'))
print(f'Found {len(csv_files)} year CSVs')

raw = (spark.read
    .option('header', True)
    .schema(raw_schema)
    .csv(csv_files)
    .withColumn('date', F.to_date('date', 'yyyy-MM-dd'))
    .withColumn('year', F.year('date'))
    .withColumn('week', F.weekofyear('date')))

# ── Join daily weather on date (broadcast — only 3653 rows) ─────────────────
weather_schema = StructType([
    StructField('date',   StringType(), True),
    StructField('ppt',    DoubleType(), True),
    StructField('tmax',   DoubleType(), True),
    StructField('tmean',  DoubleType(), True),
    StructField('tmin',   DoubleType(), True),
    StructField('vpdmax', DoubleType(), True),
    StructField('vpdmin', DoubleType(), True),
])
weather = (spark.read
    .option('header', True)
    .schema(weather_schema)
    .csv(WEATHER)
    .withColumn('date', F.to_date('date', 'yyyy-MM-dd')))

raw = raw.join(F.broadcast(weather), on='date', how='left')

print(f'Raw observations: {raw.count():,}')
raw.show(5)

Found 10 year CSVs


Raw observations: 16,291,066
+----------+-------+----------+----+----+---+------------------+-----------------+------------------+-----------------+------------------+
|      date|tile_id|      ndvi|year|week|ppt|              tmax|            tmean|              tmin|           vpdmax|            vpdmin|
+----------+-------+----------+----+----+---+------------------+-----------------+------------------+-----------------+------------------+
|2021-02-28|      0|0.31760436|2021|   8|0.0|14.594962470476041|7.817441410981431|1.0399203514868203|5.744961125906124|1.0699467390552333|
|2021-02-28|      1|0.37745973|2021|   8|0.0|14.594962470476041|7.817441410981431|1.0399203514868203|5.744961125906124|1.0699467390552333|
|2021-02-28|      2|0.40140024|2021|   8|0.0|14.594962470476041|7.817441410981431|1.0399203514868203|5.744961125906124|1.0699467390552333|
|2021-02-28|      3|0.39208633|2021|   8|0.0|14.594962470476041|7.817441410981431|1.0399203514868203|5.744961125906124|1.0699467390552333

In [4]:
weekly = (raw
    .filter(F.col('ndvi').isNotNull())
    .groupBy('tile_id', 'year', 'week')
    .agg(
        F.mean('ndvi').alias('ndvi_mean'),
        F.count('ndvi').alias('n_obs'),
        F.sum('ppt').alias('ppt_sum'),
        F.mean('tmax').alias('tmax_mean'),
        F.mean('tmean').alias('tmean_mean'),
        F.mean('tmin').alias('tmin_mean'),
        F.mean('vpdmax').alias('vpdmax_mean'),
        F.mean('vpdmin').alias('vpdmin_mean'),
    ))

print(f'Weekly rows: {weekly.count():,}')
weekly.show(5)

Weekly rows: 9,399,102


+-------+----+----+----------+-----+-------+------------------+-----------------+------------------+-----------------+------------------+
|tile_id|year|week| ndvi_mean|n_obs|ppt_sum|         tmax_mean|       tmean_mean|         tmin_mean|      vpdmax_mean|       vpdmin_mean|
+-------+----+----+----------+-----+-------+------------------+-----------------+------------------+-----------------+------------------+
|    297|2021|   8|0.28499156|    1|    0.0|14.594962470476041|7.817441410981431|1.0399203514868203|5.744961125906124|1.0699467390552333|
|    404|2021|   8| 0.2760646|    1|    0.0|14.594962470476041|7.817441410981431|1.0399203514868203|5.744961125906124|1.0699467390552333|
|    446|2021|   8|0.24499655|    1|    0.0|14.594962470476041|7.817441410981431|1.0399203514868203|5.744961125906124|1.0699467390552333|
|    473|2021|   8|0.23367004|    1|    0.0|14.594962470476041|7.817441410981431|1.0399203514868203|5.744961125906124|1.0699467390552333|
|    671|2021|   8|0.27002585|    

In [5]:
# ── NDVI anomaly: tile deviation from vineyard mean per week × year ─────────
# This strips vintage effects and isolates the spatial (terroir) signal.

vineyard_mean = (weekly
    .groupBy('year', 'week')
    .agg(F.mean('ndvi_mean').alias('vineyard_ndvi_mean')))

weekly = (weekly
    .join(vineyard_mean, on=['year', 'week'], how='left')
    .withColumn('ndvi_anomaly', F.col('ndvi_mean') - F.col('vineyard_ndvi_mean')))

print('Anomaly computed')
weekly.filter(F.col('week').between(36, 43)).show(5)


Anomaly computed


+----+----+-------+----------+-----+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+--------------------+
|year|week|tile_id| ndvi_mean|n_obs|           ppt_sum|         tmax_mean|        tmean_mean|         tmin_mean|       vpdmax_mean|       vpdmin_mean| vineyard_ndvi_mean|        ndvi_anomaly|
+----+----+-------+----------+-----+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+--------------------+
|2025|  37|    389|0.39421442|    1|1.5751937507658966|31.305036015473622|22.177524111252445|13.050012207031273|17.820154733436055|2.5936920508635346|0.41084862399084243|-0.01663420399084...|
|2025|  37|    997| 0.3147575|    1|1.5751937507658966|31.305036015473622|22.177524111252445|13.050012207031273|17.820154733436055|2.5936920508635346|0.41084862399084243|-0.09609112399084241|
|2025|  37|   1077|0.45964912|    1|1.57

In [6]:
weeks_all   = list(range(1, 52))
feature_wks = list(range(1, 36))
target_wks  = list(range(36, 44))

ndvi_wide = (weekly
    .groupBy('tile_id', 'year')
    .pivot('week', weeks_all)
    .agg(
        F.first('ndvi_mean').alias('ndvi_mean'),
        F.first('ndvi_anomaly').alias('ndvi_anomaly'),
        F.first('ppt_sum').alias('ppt_sum'),
        F.first('tmax_mean').alias('tmax_mean'),
        F.first('tmean_mean').alias('tmean_mean'),
        F.first('vpdmax_mean').alias('vpdmax_mean'),
    ))

print(f'Wide shape: {ndvi_wide.count():,} rows × {len(ndvi_wide.columns)} cols')
ndvi_wide.show(3)

26/05/19 22:15:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Wide shape: 329,780 rows × 308 cols


+-------+----+-----------+--------------------+---------+-----------------+------------------+-----------------+-----------+--------------------+---------+------------------+------------------+-----------------+-----------+--------------------+---------+------------------+-----------------+-----------------+-------------------+-------------------+---------+-----------------+-------------------+-----------------+-----------+-------------------+---------+-----------------+-------------------+------------------+-----------+--------------+---------+-----------+------------+-------------+-----------+--------------+---------+-----------+------------+-------------+-----------+-------------------+---------+-----------------+----------------+-----------------+-----------+--------------------+---------+-----------------+------------------+------------------+------------+---------------+----------+------------+-------------+--------------+------------+-------------------+------------------+------

In [7]:
from pyspark.sql import functions as F
from functools import reduce

feat_mean_cols    = [f'{w}_ndvi_mean'    for w in feature_wks if f'{w}_ndvi_mean'    in ndvi_wide.columns]
target_anom_cols  = [f'{w}_ndvi_anomaly' for w in target_wks  if f'{w}_ndvi_anomaly' in ndvi_wide.columns]

# Mean of available target weeks — coalesce nulls to 0 and divide by count of
# non-null weeks. Rows where no target week has an observation stay null (dropped later).
anom_sum   = reduce(lambda a, b: a + b, [F.coalesce(F.col(c), F.lit(0.0)) for c in target_anom_cols])
anom_count = reduce(lambda a, b: a + b, [F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in target_anom_cols])
ndvi_wide = ndvi_wide.withColumn(
    'ndvi_anomaly_harvest',
    F.when(anom_count > 0, anom_sum / anom_count).otherwise(F.lit(None).cast('double'))
)

# Season health: mean NDVI over weeks 28-43 (same null-safe approach)
season_mean_cols = [f'{w}_ndvi_mean' for w in range(28, 44) if f'{w}_ndvi_mean' in ndvi_wide.columns]
seas_sum   = reduce(lambda a, b: a + b, [F.coalesce(F.col(c), F.lit(0.0)) for c in season_mean_cols])
seas_count = reduce(lambda a, b: a + b, [F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in season_mean_cols])
ndvi_wide = ndvi_wide.withColumn(
    'ndvi_season_mean',
    F.when(seas_count > 0, seas_sum / seas_count).otherwise(F.lit(None).cast('double'))
)

print('Feature summary done')

Feature summary done


In [8]:
# ── Join terrain features (static per tile — broadcast join) ──────────────
import pandas as pd

terrain_pd = pd.read_csv(TERRAIN)
terrain_sp = spark.createDataFrame(terrain_pd).cache()
print(f'Terrain: {terrain_sp.count():,} rows × {len(terrain_sp.columns)} cols')

wide = ndvi_wide.join(F.broadcast(terrain_sp), on='tile_id', how='left')


Terrain: 32,978 rows × 23 cols


In [9]:
soil_pd = pd.read_csv(SOIL)
soil_sp = spark.createDataFrame(soil_pd).cache()
print(f'Soil: {soil_sp.count():,} rows × {len(soil_sp.columns)} cols')

wide = wide.join(F.broadcast(soil_sp), on='tile_id', how='left')

print(f'Assembled: {wide.count():,} rows × {len(wide.columns)} cols')

Soil: 32,978 rows × 24 cols


Assembled: 329,780 rows × 355 cols


In [10]:
# ── Save as Parquet (efficient for subsequent MLlib reads) ──────────────────
wide.write.mode('overwrite').parquet(OUT_DIR)
print(f'Saved to {OUT_DIR}')


Saved to ../data/spark/assembled


In [11]:
check = spark.read.parquet(OUT_DIR)
print(f'Parquet rows: {check.count():,}')
print('Columns:', check.columns[:20], '...')
check.select(
    'tile_id', 'year', 'ndvi_anomaly_harvest',
    'elev_mean', 'ph1to1h2o_r',
    '20_ppt_sum', '20_tmax_mean', '20_vpdmax_mean',
).show(5)

Parquet rows: 329,780
Columns: ['tile_id', 'year', '1_ndvi_mean', '1_ndvi_anomaly', '1_ppt_sum', '1_tmax_mean', '1_tmean_mean', '1_vpdmax_mean', '2_ndvi_mean', '2_ndvi_anomaly', '2_ppt_sum', '2_tmax_mean', '2_tmean_mean', '2_vpdmax_mean', '3_ndvi_mean', '3_ndvi_anomaly', '3_ppt_sum', '3_tmax_mean', '3_tmean_mean', '3_vpdmax_mean'] ...
+-------+----+--------------------+------------------+-----------------+----------+-----------------+------------------+
|tile_id|year|ndvi_anomaly_harvest|         elev_mean|      ph1to1h2o_r|20_ppt_sum|     20_tmax_mean|    20_vpdmax_mean|
+-------+----+--------------------+------------------+-----------------+----------+-----------------+------------------+
|     18|2019|-0.16906299074213715| 204.6023937127529|              7.0|      NULL|             NULL|              NULL|
|     58|2021|-0.00228300048970...|206.66865343150525|              7.0|       0.0|28.09496247047604|17.089922871700555|
|     82|2021|0.008078963796013337| 207.6602845144744|    